# 01 · export v1 — Z: pickles → canonical CSV

**Kernel: `fttl-v1` (env-v1, Python 3.5).** Reads v1's surviving data artefacts off the `Z:`
drive and writes them into `src/data/real/` under canonical names, so the analysis `.venv`
never has to open a pandas-0.22 pickle.

Two 3.5-era constraints shape this file:
- **`src/config.py` cannot be imported here** (it needs Python 3.7+), so the paths and column
  names below MIRROR `config.VERSIONS["v1"]` by hand. If config changes, change here too.
- **This pandas cannot write parquet** — output is CSV; the last cell shows the one-liner that
  converts CSV → parquet in the analysis env.

| writes | canonical columns | config kind |
|---|---|---|
| `inputs/raw_v1.csv` | raw extract as-is, id renamed | `raw_dataset` |
| `inputs/features_v1.csv` | `claim_id` + transformed matrix (target kept; set aside downstream) | `processed_inputs` |
| `inputs/targets_v1.csv` | `claim_id, date, observed` | `targets` |
| `detection/v1_scores.csv` | `claim_id, model_v1_score` | `scores` (train-time; in-sample for the train split) |


In [ ]:
# -*- Python 3.5 compatible: no f-strings -*-
import os
import sys

import pandas as pd

assert sys.version_info[:2] == (3, 5), "run this on the fttl-v1 kernel (env-v1)"

# ---- SOURCES: fill in the real Z: paths. These live HERE, not in config — a declared config
# ---- path would point the analysis .venv at a pickle it cannot open. -------------------
RAW_DATASET      = r"Z:\P10_...\inputs.pkl"              # raw extract
PROCESSED_INPUTS = r"Z:\...\inputs_transformed.pkl"      # 4 splits appended, post-preprocessing
PREDICTIONS      = r"Z:\...\predictions.pkl"             # [claimnumber, predictions]

# ---- canonical names (mirror of config.VERSIONS['v1']['columns']) ----
ID, DATE, OBSERVED, SCORE = "claimnumber", "lossdate", "veh_total_loss", "predictions"

ROOT = os.getcwd()
while not os.path.exists(os.path.join(ROOT, "src", "config.py")):
    parent = os.path.dirname(ROOT)
    assert parent != ROOT, "repo root not found above cwd"
    ROOT = parent
OUT = os.path.join(ROOT, "src", "data", "real")
print("repo root:", ROOT)


In [ ]:
raw  = pd.read_pickle(RAW_DATASET)
proc = pd.read_pickle(PROCESSED_INPUTS)
pred = pd.read_pickle(PREDICTIONS)
print("raw        ", raw.shape)
print("processed  ", proc.shape)
print("predictions", pred.shape)
print("\nprocessed columns:", list(proc.columns))

# the transformed table should carry the id (39 model-ready cols include claimnumber + target)
assert ID in proc.columns, "no '" + ID + "' in the transformed table — inspect and adjust"
assert ID in pred.columns and SCORE in pred.columns


In [ ]:
def out_path(sub, name):
    d = os.path.join(OUT, sub)
    if not os.path.isdir(d):
        os.makedirs(d)
    return os.path.join(d, name)

raw.rename(columns={ID: "claim_id"}).to_csv(out_path("inputs", "raw_v1.csv"), index=False)

feats = proc.rename(columns={ID: "claim_id"})
feats.to_csv(out_path("inputs", "features_v1.csv"), index=False)

targets = raw[[ID, DATE, OBSERVED]].rename(
    columns={ID: "claim_id", DATE: "date", OBSERVED: "observed"})
targets.to_csv(out_path("inputs", "targets_v1.csv"), index=False)

scores = pred[[ID, SCORE]].rename(columns={ID: "claim_id", SCORE: "model_v1_score"})
scores.to_csv(out_path("detection", "v1_scores.csv"), index=False)

print("raw {0} / features {1} / targets {2} / scores {3}".format(
    len(raw), len(feats), len(targets), len(scores)))
print("claim_id overlap features-scores:", feats["claim_id"].isin(scores["claim_id"]).mean())


## Then, in the analysis `.venv` — CSV → parquet

```python
import pandas as pd
for src, dst in [("inputs/raw_v1.csv",       "inputs/raw_v1.parquet"),
                 ("inputs/features_v1.csv",  "inputs/features_v1.parquet"),
                 ("inputs/targets_v1.csv",   "inputs/targets_v1.parquet"),
                 ("detection/v1_scores.csv", "detection/v1_scores.parquet")]:
    pd.read_csv("src/data/real/" + src).to_parquet("src/data/real/" + dst, index=False)
```

These parquet paths ARE config's fallback locations, so `loaders.load("v1")` resolves everything
with no config declaration.

Notes:
- The train split's scores are **in-sample**; carry that caveat into anything that uses them.
- Split membership is reconstructable from `date` against 2017-04-01 / 2017-06-30 boundaries
  (the 80/20 `train_test_split` within the first window was random — NOT reconstructable).
- `cc_fttl`-flagged rows (~2.5%) were EXCLUDED from training; the raw table still has them.
